# Тестирование границ аргументов у BaseTransform

Ноутбук сделан не для примера, а для тестирования правильно указанных границ в `BaseTransform.get_ranges()`.

In [1]:
import inspect
import os
import numpy as np
from typing import Any
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")


%load_ext autoreload
%autoreload 2

In [2]:
if 'notebooks' in os.listdir():
    pass
else:
    os.chdir('..')

print(os.getcwd())

C:\Users\kobel\учёба\diploma\fas_aug_attack


## Импорт всех transforms

In [3]:
from src import transforms
from src.base import BaseTransform

In [4]:
all_members = inspect.getmembers(transforms)

list_type_transforms = [
    cls for name, cls in all_members
    if inspect.isclass(cls) 
    and issubclass(cls, transforms.BaseTransform) 
    and cls is not BaseTransform
]
print(f'Импортировано классов = {len(list_type_transforms)}, наследованных от `BaseTransform`')
list_type_transforms[:5]

Импортировано классов = 45, наследованных от `BaseTransform`


[src.transforms.transforms.BlurTransform,
 src.transforms.transforms.BrightnessContrastTransform,
 src.transforms.transforms.CLAHETransform,
 src.transforms.transforms.ChannelShuffleTransform,
 src.transforms.transforms.ChromaticAberrationTransform]

## Дополнительная настройка для `Pipeline`

In [5]:
from src.base import BaseModel


class RandomModel(BaseModel):
    def __init__(self, model: Any):
        super().__init__(model=model)

    def predict(self, img: Any) -> float:
        return np.random.rand()

    
model = RandomModel(model=None)
model.predict(1)

0.08806123966168344

In [6]:
from src.utils.dataset import AttackDataset


dataset = AttackDataset('notebooks/example_dataset')
dataset[0].keys()

dict_keys(['img', 'filename', 'path2file', 'is_real', 'type_attack'])

## Запуск тестов границ аргументов у transforms

In [11]:
from src.pipeline import PipelineAttackOptunaImg

In [12]:
ALL_MINUTES = 20


timeout = (ALL_MINUTES * 60) / len(list_type_transforms)

In [13]:
for i in tqdm(range(len(list_type_transforms))):

    optuna_attack_pipeline = PipelineAttackOptunaImg(
        model=model,
        list_type_transforms=[list_type_transforms[i]],
    )
    
    optuna_attack_pipeline.optimize(
        data=dataset[0]['img'],
        direction='minimize',
        n_trials=None,
        timeout=timeout,
        show_progress=False,
    )

100%|████████████████████████████████████████████████| 45/45 [20:08<00:00, 26.86s/it]
